# ZhongJing-TCM Benchmark · 本地 HuggingFace 医学/中医模型评测（Colab）

用 **Kimi-K2.6（Azure）生成的中医基准**，对一批开源 **HuggingFace 医学/中医大模型**做本地评测。
出题在 [`colab_azure_generation.ipynb`](colab_azure_generation.ipynb) 完成；本 notebook 专注 **M8 评测 + M9 统计 + 临床评测框架**。

| 能力 | 实现 |
|---|---|
| 🤗 **本地推理** | 新增 `local` provider：`transformers` + `apply_chat_template`，按需 4-bit 量化（bitsandbytes） |
| 🔁 **单卡轮转** | 评测逐个模型进行，**同一时刻只驻留一个模型**，切换时自动释放显存 |
| 📚 **模型注册表** | `configs/hf_models.yaml` 维护 友好名→repo+加载参数（量化/dtype/trust_remote_code/多模态/聊天模板回退） |
| 🖼️ **多模态文本评测** | `Lingshu`、`medgemma-27b-it` 等 VL 模型经 `AutoProcessor` 仅用文本通道评测 |
| 🔐 **门控模型** | `Baichuan-M1`、`google/medgemma-27b-it` 需在 HF 接受协议并提供 `HF_TOKEN` |

**接入模型（16，注册表 `configs/hf_models.yaml`）**：HuatuoGPT-o1、HuatuoGPT-II、ClinicalGPT-R1、Aquila-Med、Taiyi、BianCang、BioMistral、CMLM-Dao1(MoE)、DISC-MedLLM、Baichuan-M1、Baichuan-M2、DeepSeek-R1-32B、Lingshu、google/medgemma-27b-it、Citrus、Meditron-70B。

> ⚠️ **显存现实**：建议 **A100-40GB**。7–8B 可 bf16 直跑；13–34B 用 4-bit；**70B（Citrus / Meditron-70B）需 A100-80GB 或多卡**。L4-24GB 只建议跑 7–8B。
> 💡 `Qilin-Med`（仅 LLaVA 视觉版）与 `Qibo`（论文版，无公开权重）未在 HF 提供文本权重，已从注册表排除。

## 1. 检查 GPU 与安装依赖

In [ ]:
#@title GPU 信息 + 安装推理依赖
!nvidia-smi -L || echo "⚠️ 未检测到 GPU：请在 代码执行程序 → 更改运行时类型 选择 A100/L4 GPU"

# 本地推理核心
!pip install -q -U "transformers>=4.52" accelerate bitsandbytes sentencepiece einops safetensors
# 多模态(VL)与部分自定义架构需要
!pip install -q -U "qwen-vl-utils" timm torchvision pillow
# 流水线/数据/评测依赖（与生成 notebook 一致）
!pip install -q openai tenacity diskcache "pydantic>=2" pyyaml tiktoken jieba pandas numpy scikit-learn typer tqdm chardet json-repair statsmodels matplotlib

import torch
print("✅ torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))

## 2. 获取代码

In [ ]:
#@title 克隆仓库并切到目标分支
import os, sys

REPO_URL = "https://github.com/pariskang/ZhongJing-TCM-Benchmark.git"  #@param {type:"string"}
BRANCH   = "claude/wizardly-ride-yryr5w"  #@param {type:"string"}
WORKDIR  = "/content/ZhongJing-TCM-Benchmark"

if not os.path.isdir(WORKDIR):
    !git clone -q "$REPO_URL" "$WORKDIR"
%cd "$WORKDIR"
!git fetch -q origin "$BRANCH" && git checkout -q "$BRANCH" && git pull -q origin "$BRANCH"

if os.path.join(WORKDIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(WORKDIR, "src"))
print("✅ 代码就绪:", WORKDIR)

## 3.（推荐）用 Google Drive 缓存模型权重与持久化结果

HF 模型动辄几十 GB，下载慢。把 `HF_HOME` 指到 Drive，**模型只下一次**，跨重连复用；
同时把 `results/`、`data/final` 重定向到 Drive 以保留评测产物与基准数据。

In [ ]:
#@title 挂载 Drive：缓存权重 + 持久化结果
USE_DRIVE = True   #@param {type:"boolean"}
PERSIST   = "/content/drive/MyDrive/zhongjing_tcm"  #@param {type:"string"}

import os, pathlib
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    hf_home = os.path.join(PERSIST, "hf_home")
    os.makedirs(hf_home, exist_ok=True)
    os.environ["HF_HOME"] = hf_home               # 模型/数据集缓存目录
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1" # 更快下载（若已装 hf_transfer）
    print("✅ HF_HOME ->", hf_home)

    def _link_to_drive(sub):
        src = pathlib.Path(WORKDIR) / sub
        dst = pathlib.Path(PERSIST) / sub
        dst.mkdir(parents=True, exist_ok=True)
        if src.is_symlink():
            src.unlink()
        elif src.exists():
            import shutil
            for f in src.glob("*"):
                tgt = dst / f.name
                if not tgt.exists():
                    shutil.move(str(f), str(tgt))
            shutil.rmtree(src, ignore_errors=True)
        src.parent.mkdir(parents=True, exist_ok=True)
        src.symlink_to(dst, target_is_directory=True)
    for sub in ["data/final", "results"]:
        _link_to_drive(sub)
    print("✅ data/final 与 results 已重定向到 Drive")
else:
    print("ℹ️ 未启用 Drive：权重每次重连都会重新下载，结果存于 /content。")

## 4. HuggingFace 登录（门控模型必需）

`Baichuan-M1`、`google/medgemma-27b-it` 是**门控模型**：先在其 HF 页面点击接受协议，
再用具有访问权限的 token 登录。其余模型无需登录。

In [ ]:
#@title 填入 HF Token（门控模型）
HF_TOKEN = ""  #@param {type:"string"}
import os
if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN.strip()
    try:
        from huggingface_hub import login
        login(HF_TOKEN.strip())
        print("✅ 已登录 HuggingFace")
    except Exception as e:
        print("⚠️ 登录失败（仍写入环境变量）：", e)
else:
    print("ℹ️ 未填 token：仅能评测非门控模型（跳过 Baichuan-M1 / medgemma）。")

## 5. 选择被测模型与加载选项

从 `configs/hf_models.yaml` 注册表里勾选要评测的模型。`provider=local`，模型逐个轮转、用完即释放显存。
- `GLOBAL_QUANT`：留空则用注册表中每个模型的默认量化；填 `4bit` 可强制全部 4-bit（小显存救命）。
- `MAX_NEW_TOKENS`：推理/思维链模型（HuatuoGPT-o1 / DeepSeek-R1）给足空间（≥2048）。

In [ ]:
#@title 配置本地评测
import os, yaml
from pathlib import Path
from llm_client import load_local_registry

PROVIDER       = "local"  #@param ["local"]
GLOBAL_QUANT   = ""       #@param ["", "none", "4bit", "8bit"]
MAX_NEW_TOKENS = 2048     #@param {type:"integer"}
N_EVAL_RUNS    = 1        #@param {type:"integer"}
TEMPERATURE    = 0.3      #@param {type:"number"}

# 勾选要评测的模型（默认一组能在 A100-40GB 跑通的 7–34B；70B 默认不选）
MODELS = "HuatuoGPT-o1, ClinicalGPT-R1, BianCang, BioMistral, Aquila-Med, Taiyi, HuatuoGPT-II, DISC-MedLLM, Baichuan-M2, DeepSeek-R1-32B"  #@param {type:"string"}

os.environ["ZHONGJING_LLM_PROVIDER"] = PROVIDER
if GLOBAL_QUANT.strip():
    os.environ["HF_QUANT"] = GLOBAL_QUANT.strip()
elif "HF_QUANT" in os.environ:
    del os.environ["HF_QUANT"]

reg = load_local_registry()
selected = [m.strip() for m in MODELS.split(",") if m.strip()]
unknown = [m for m in selected if m not in reg]
assert not unknown, f"注册表中没有这些模型: {unknown}\n可选: {list(reg)}"

# 写入 pipeline.yaml：provider=local + 被测模型 + 评测参数
cfg_path = Path(WORKDIR) / "configs" / "pipeline.yaml"
conf = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
conf["llm"]["provider"] = PROVIDER
conf["llm"]["max_tokens"] = int(MAX_NEW_TOKENS)
conf["llm"]["temperature"] = float(TEMPERATURE)
conf["evaluate"]["models"] = selected
conf["evaluate"]["n_runs"] = int(N_EVAL_RUNS)
# 简答题语义判分若想用某个强模型，可设 evaluate.judge_model；这里默认用各自被测模型。
cfg_path.write_text(yaml.safe_dump(conf, allow_unicode=True, sort_keys=False), encoding="utf-8")

import pandas as pd
rows = [{"模型": m, "repo": reg[m]["repo"], "参数": reg[m].get("params","?"),
         "量化": (GLOBAL_QUANT or reg[m].get("quant","none")),
         "VL": reg[m].get("loader","-"), "门控": reg[m].get("gated", False)} for m in selected]
print(f"✅ 已选 {len(selected)} 个模型（provider=local, n_runs={N_EVAL_RUNS}）")
pd.DataFrame(rows)

## 6. 准备评测基准（`data/final`）

M8 读取 `data/final/zhongjing_tcm_diagnostic.jsonl`。三种来源：
- `drive`：已由 Azure 生成并存于 Drive（开启第 3 步即自动可见）——**推荐**
- `mock_quick`：用 **mock** 出题器在仓库自带 3 篇文章上跑 M1→M7，**免费/免 GPU** 快速造一个小基准（仅供打通流程）
- `existing`：`data/final` 已就位，直接用

In [ ]:
#@title 准备/检查基准数据集
DATA_SOURCE = "mock_quick"  #@param ["drive", "mock_quick", "existing"]
from pathlib import Path
from config import load_config
load_config.cache_clear()
cfg = load_config()
final_dir = cfg.path("paths.final_dir")
diag = final_dir / "zhongjing_tcm_diagnostic.jsonl"
full = final_dir / "zhongjing_tcm_full.jsonl"

def _have(): return diag.exists() or full.exists()

if DATA_SOURCE == "existing":
    assert _have(), f"未找到 {diag} 或 {full}"
elif DATA_SOURCE == "drive":
    assert _have(), "Drive 中未发现基准；请先用 Azure notebook 生成，或改用 mock_quick。"
elif DATA_SOURCE == "mock_quick" and not _have():
    import os
    os.environ["ZHONGJING_LLM_PROVIDER"] = "mock"   # 仅出题用 mock；评测仍用本地模型
    import m1_ingest, m2_quality, m4_label, m5_generate, m6_dtqf, m7_assemble
    from collections import Counter
    from schemas import Article
    from utils import load_jsonl_as, save_jsonl
    from m3_topic import chunk_article, jieba_tok
    load_config.cache_clear()
    m1_ingest.run()
    m2_quality.run(llm_judge=False)
    interim = cfg.path("paths.interim_dir")
    arts = [a for a in load_jsonl_as(interim / "articles_scored.jsonl", Article) if a.quality_passed]
    passages = []
    for tid, a in enumerate(arts):
        chunks = chunk_article(a, max_len=cfg.get("topic.max_len",400),
                               overlap=cfg.get("topic.overlap",80), min_len=cfg.get("topic.min_passage_len",50))
        kw = [w for w,_ in Counter(jieba_tok(a.clean_text)).most_common(10)]
        for p in chunks: p.topic_id, p.topic_keywords = tid, kw
        passages.extend(chunks)
    save_jsonl(passages, interim / "passages_topiced.jsonl")
    m4_label.run(); m5_generate.run(); m6_dtqf.run(); m7_assemble.run()
    os.environ["ZHONGJING_LLM_PROVIDER"] = "local"  # 复位到本地评测
    load_config.cache_clear()

path = diag if diag.exists() else full
import json
n = sum(1 for _ in open(path, encoding="utf-8")) if _have() else 0
print(f"✅ 基准就绪：{path.name}（{n} 题）" if n else "❌ 仍无基准数据，请检查上面来源。")

## 7. 连通性自检（加载第一个模型，试跑一题）

首次会**下载权重**（大模型较慢）。成功打印一段中文回答即说明本地推理链路通。

In [ ]:
#@title 加载最小模型并试跑
from llm_client import call_text
m0 = selected[0]
print(f"⏳ 正在加载 {m0}（{reg[m0]['repo']}）… 首次需下载权重")
try:
    out = call_text("用一句话解释中医“辨证论治”。", model=m0, use_cache=False, max_tokens=256)
    print(f"\n✅ {m0} 响应：\n{out[:400]}")
except Exception as e:
    print(f"❌ {m0} 失败：{e!r}\n（门控模型需 HF_TOKEN；显存不足试 GLOBAL_QUANT=4bit；自定义架构需联网装 trust_remote_code 代码）")

## 8. ⭐ 逐模型评测（M8）

对每个被测模型在诊断子集上做 STAGER 零样本评测（选择题 Acc/P/R/F1 + 简答题语义判分），
**逐个加载→评测→释放显存**。结果写入 `results/eval_<model>.jsonl` 与 `results/metrics.csv`。
中断后重跑会跳过已完成的模型（命中缓存）。

In [ ]:
#@title 逐模型评测
import gc, torch, traceback
import m8_evaluate
from llm_client import get_client
from config import load_config
load_config.cache_clear()

all_metrics = []
for m in selected:
    print(f"\n========== 评测 {m}（{reg[m]['repo']}）==========")
    try:
        all_metrics.append(m8_evaluate.run(m))
        print("  指标:", all_metrics[-1])
    except Exception as e:
        print(f"  ⏭️ 跳过 {m}：{e!r}")
        traceback.print_exc()
    finally:
        # 释放当前模型显存，准备下一个
        try:
            get_client()._hf_slot = None
        except Exception:
            pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\n📊 完成模型数:", len(all_metrics))

## 9. M9 统计 + 模型对比汇总

In [ ]:
#@title 统计 + 对比表/柱状图
import m9_stats, pandas as pd, matplotlib.pyplot as plt
from config import load_config
load_config.cache_clear()
try:
    stats = m9_stats.run()
    print("📈 统计:", list(stats.keys()) if stats else "（无记录）")
except Exception as e:
    print("⏭️ 统计跳过：", e)

cfg = load_config()
csv = cfg.path("paths.results_dir") / "metrics.csv"
if csv.exists():
    df = pd.read_csv(csv)
    cols = [c for c in ["model","n","accuracy","precision","recall","f1","refusal_rate"] if c in df.columns]
    df = df[cols].sort_values("accuracy", ascending=False) if "accuracy" in df.columns else df
    display(df)
    if "accuracy" in df.columns and not df.empty:
        plt.rcParams["font.sans-serif"] = ["Noto Sans CJK SC","DejaVu Sans"]
        ax = df.plot.barh(x="model", y="accuracy", legend=False, figsize=(8, 0.5*len(df)+1))
        ax.set_xlabel("Accuracy"); ax.set_title("各医学模型 · 诊断子集准确率"); plt.tight_layout(); plt.show()
else:
    print("（暂无 metrics.csv）")

## 10.（可选）临床评测框架 on 本地模型

对单个本地模型跑 T0–T6 / L1–L4 的小型 demo（详见 `docs/CLINICAL_EVAL_FRAMEWORK.md`）。
逐 tier 容错；需要 M4/M7 产物的 tier（invariance/counterfactual）依赖第 6 步。

In [ ]:
#@title 在一个本地模型上跑临床评测框架
RUN_FRAMEWORK = False  #@param {type:"boolean"}
FW_MODEL = selected[0]  #@param {type:"string"}
if RUN_FRAMEWORK:
    from config import load_config; load_config.cache_clear()
    import m8_evaluate, t1_counterfactual, t2_patient_sim, t3_tools, t4_longitudinal
    import t5_mdt, t6_dialogue, l2_process, l3l4_rubric, abstention, calibration, judges as _judges
    _fw = {}
    def _run(name, fn):
        try: _fw[name]=fn(); print("✅", name)
        except Exception as e: _fw[name]=f"skip({e})"; print("⏭️", name, "—", e)
    _run("invariance·T0", lambda: m8_evaluate.run_invariance(FW_MODEL))
    _run("counterfactual·T1", lambda: t1_counterfactual.run(model=FW_MODEL, limit=3))
    _run("consult·T2", lambda: t2_patient_sim.run(model=FW_MODEL, max_turns=6))
    _run("tools·T3", lambda: t3_tools.run(model=FW_MODEL))
    _run("episode·T4", lambda: t4_longitudinal.run(model=FW_MODEL))
    _run("mdt·T5", lambda: t5_mdt.run(model=FW_MODEL))
    _run("dialogue·T6", lambda: t6_dialogue.run(model=FW_MODEL))
    _run("process·L2", lambda: l2_process.run(model=FW_MODEL))
    _run("rubric·L3L4", lambda: l3l4_rubric.run(model=FW_MODEL))
    _run("abstain·A@D", lambda: abstention.run(model=FW_MODEL))
    _run("calibrate", lambda: calibration.run(model=FW_MODEL))
    _run("judges", lambda: _judges.run(model=FW_MODEL))
    print("\n🧪 完成:", {k:(v if isinstance(v,str) else "ok") for k,v in _fw.items()})
else:
    print("已跳过临床评测框架。")

## 11. 导出结果

In [ ]:
#@title 打包并下载 results
import os
os.chdir(WORKDIR)
!zip -qr /content/zhongjing_hf_eval.zip results data/final -x "*.gitkeep"
print("✅ -> /content/zhongjing_hf_eval.zip")
try:
    from google.colab import files
    files.download("/content/zhongjing_hf_eval.zip")
except Exception as e:
    print("（非 Colab，跳过下载）", e)

---
### 小贴士
- **显存不足 / OOM**：`GLOBAL_QUANT=4bit`；或减少 `MAX_NEW_TOKENS`；或只选 7–8B 模型。70B 需 A100-80GB/多卡。
- **门控模型**：`Baichuan-M1`、`google/medgemma-27b-it` 必须先在 HF 接受协议并填 `HF_TOKEN`。
- **自定义架构**：`trust_remote_code` 默认开启；Baichuan/Aquila/Taiyi/HuatuoGPT-II 会联网拉取建模代码。
- **多模态模型**：`Lingshu`、`medgemma` 经 `AutoProcessor` 仅走文本通道评测（不喂图像）。
- **聊天模板**：自定义格式模型（HuatuoGPT-II/Taiyi/DISC-MedLLM）用注册表里的回退模板，保真度尽力而为。
- **加新模型**：往 `configs/hf_models.yaml` 加一条 `友好名: {repo, quant, trust_remote_code, ...}` 即可，无需改代码。
- **省 GPU**：先用 `mock_quick` 造小基准 + 只选 1 个 7B 模型打通，再扩展。